In [1]:
import mon2y
import optuna
import math
import pandas as pd
import numpy as np
import logging
from IPython.display import display

/home/lachlan/mon2y_rs/env/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
logging.basicConfig(
    format='%(asctime)s %(levelname)s %(message)s',
    datefmt='%Y-%m-%d %H:%M:%S',
    level=logging.INFO
)

In [3]:
target_game_turns = 25
max_iterations = 30000
trials = 100

study = optuna.create_study(
    storage="sqlite:///db.sqlite3",  # Specify the storage URL here.
    study_name="connect4_25_turns_1",
    directions = ["minimize", "minimize", "minimize"],
    load_if_exists = True,
)

[I 2025-12-24 15:27:54,808] A new study created in RDB with name: connect4_25_turns_1


In [4]:
def objective(trial: optuna.Trial):
    board_width = trial.suggest_int("board_width", 4, 100)
    board_height = trial.suggest_int("board_height", 4, 100)

    explore_iterations = max_iterations*(1-(math.log(max(1,trials - max(trial.number, 2))) / math.log(trials)))
    logging.info(f"iterations: {explore_iterations} board_width: {board_width} board_height: {board_height}")
    
    raw_results = mon2y.explore(mon2y.Games.C4, int(explore_iterations), 8, hyperparams={"board_width":board_width,"board_height":board_height})
    logging.info("  explore done")
    
    df = pd.DataFrame(raw_results)
    logging.info("  dataframe converted")
    df['ratio']= (df['turns']-df['rwalk'])/df['turns']
    df['norm_sum_diff_est_reward'] = (df['sum_diff_est_reward'] - df['sum_diff_est_reward'].min()) / (df['sum_diff_est_reward'].max() - df['sum_diff_est_reward'].min())
    df['trust'] = df['ratio'] * df['norm_sum_diff_est_reward']
    df['norm_trust'] = (df['trust']-df['trust'].min()) / (df['trust'].max() - df['trust'].min())
    
    w = df['norm_trust']
    Neff = w.sum()
    
    # ---- Turns ----
    mu_T = (w * df['turns']).sum() / Neff
    var_T = (w * (df['turns'] - mu_T)**2).sum() / Neff
    se_T = np.sqrt(var_T / Neff)
    z_T = (mu_T - target_game_turns)
    
    # ---- Win-rate ----
    wins = (df['winning_player'] == 1).astype(float)
    p_hat = (w * wins).sum() / Neff
    se_p = np.sqrt(p_hat * (1 - p_hat) / Neff)
    z_p = (p_hat - 0.5)

    logging.info("Results calculated")
    return abs(z_T), abs(z_p), 0.01*((abs(board_width - 8) * abs(board_height-8)))

In [5]:
study.optimize(objective, n_trials=trials)
print(f"Best value: {study.best_value} (params: {study.best_params})")


2025-12-24 15:27:59 INFO iterations: 131.6088646125768 board_width: 74 board_height: 36
2025-12-24 15:28:06 INFO   explore done
2025-12-24 15:28:06 INFO   dataframe converted
2025-12-24 15:28:06 INFO Results calculated
[I 2025-12-24 15:28:06,827] Trial 0 finished with values: [35.098581199936234, 0.06226427492105063, 18.48] and parameters: {'board_width': 74, 'board_height': 36}.
2025-12-24 15:28:06 INFO iterations: 131.6088646125768 board_width: 71 board_height: 86
2025-12-24 15:28:23 INFO   explore done
2025-12-24 15:28:23 INFO   dataframe converted
2025-12-24 15:28:23 INFO Results calculated
[I 2025-12-24 15:28:23,118] Trial 1 finished with values: [32.14363333754255, 0.020920552336040843, 49.14] and parameters: {'board_width': 71, 'board_height': 86}.
2025-12-24 15:28:23 INFO iterations: 131.6088646125768 board_width: 63 board_height: 37
2025-12-24 15:28:28 INFO   explore done
2025-12-24 15:28:28 INFO   dataframe converted
2025-12-24 15:28:28 INFO Results calculated
[I 2025-12-24 1

RuntimeError: A single best trial cannot be retrieved from a multi-objective study. Consider using Study.best_trials to retrieve a list containing the best trials.